# Aggregazione stagionale delle previsioni decadali

Dai file mensili di ogni inizializzazione e membro
(`CONFESS_DATA/<exp>/1x1/monthly_start_dates/<var>/`) produce le medie
stagionali (DJF, MAM, JJA, SON) per lead singolo e per combinazione di lead
year, con gli stessi nomi di file dell'originale
`00-process_2025/03-decadal_season_aggregation`. Li leggono `seasons/`
(bias, ACC, serie temporali) e `cover_mechanism/`. Funzioni e dettagli in
`season_aggregation_lib.py`.

Rispetto all'originale:

- **date**: ogni file mensile viene aperto con `af.fix_monthly_time`. I file
  non CMOR di SENS (per esempio la neve, `snd_s1999-r1i1p1f1_199911-200010_1x1.nc`)
  sono datati al primo giorno del mese dopo, e l'originale li raggruppava per
  data: le loro stagioni erano sfasate di un mese. I file CMOR non cambiano;
- le medie su più lead year usano solo le inizializzazioni con tutti gli anni;
- esecuzione in processi `spawn` freschi, come nel resto del repo.

Si lancia una variabile e un esperimento alla volta (`exp`, `var` nella cella
sotto), come l'originale. I file già presenti con gli stessi nomi vengono
sovrascritti.

In [ ]:
# rende config.py (in notebooks/) importabile anche da questa sottocartella
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)

exp = 'a52o'
var = 'snd'
REALM = None                      # None: quello dei file stagionali gia' presenti (cosi' si sovrascrivono); se non ce ne sono, 'Amon'
START_YEARS = range(1993, 2014)   # inizializzazioni, come l'originale
MEMBERS = [f'r{m}i1p1f1' for m in range(1, 11)]
N_PROC = 10                       # processi in parallelo (uno per membro)

In [ ]:
sys.path.insert(0, os.getcwd())
import season_aggregation_lib as sal

realms = sal.check_inputs(exp, var, START_YEARS, MEMBERS)
if REALM is None:
    if len(realms) > 1:
        raise ValueError(f"file stagionali gia' presenti con piu' realm ({realms}): "
                         "togliere quelli vecchi prima di rifarli, altrimenti chi li legge trova doppioni")
    REALM = realms[0] if realms else 'Amon'
print('realm nei nomi dei file:', REALM)

In [ ]:
%%time
import multiprocessing as mp

# medie stagionali per inizializzazione e anno, serie per lead, medie sui lead year (un processo per membro)
jobs = [(exp, var, REALM, m, list(START_YEARS)) for m in MEMBERS]
with mp.get_context('spawn').Pool(processes=N_PROC, maxtasksperchild=1) as pool:
    for _i, msg in enumerate(pool.imap_unordered(sal.run_member, jobs), 1):
        print(f"  {msg}   {_i}/{len(jobs)}", flush=True)

In [ ]:
%%time
# ensemble dei membri per ogni combinazione di lead year (un processo per stagione)
jobs = [(exp, var, REALM, season) for season in sal.SEASONS]
with mp.get_context('spawn').Pool(processes=min(N_PROC, len(jobs)), maxtasksperchild=1) as pool:
    for _i, msg in enumerate(pool.imap_unordered(sal.run_ensemble, jobs), 1):
        print(f"  {msg}   {_i}/{len(jobs)}", flush=True)

Dopo averla rifatta per una variabile, vanno rifatti i calcoli che la
leggono. Per la neve di SENS (`exp='a52o'`, `var='snd'`):
`cover_mechanism/01-mean_state_delta` con `MODEL_VARS = ['snd']` (solo la
cella del modello), poi `cover_mechanism/01-Fig_mean_state_delta`.